In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure the plotting routines
import pandas as pd

# Import the CRUD module developed in Project One
from CRUD_Python_Module import CRUD

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD module
db = CRUD(username, password)

# Retrieve all records for the initial unfiltered dashboard view
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=["_id"], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo and unique identifier
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())

app.layout = html.Div([
    html.Center([
        html.B(html.H1("Grazioso Salvare Rescue Dashboard")),
        html.P("Created by Michael", style={"fontStyle": "italic", "fontSize": "18px"}),
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image.decode()),
                style={"height": "140px"},
            ),
            href="http://www.snhu.edu",
            target="_blank",
        ),
    ]),
    html.Hr(),
    html.Div([
        html.H3("Select Rescue Training Filter"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": " Water Rescue", "value": "water_rescue"},
                {
                    "label": " Mountain or Wilderness Rescue",
                    "value": "mountain_wilderness_rescue",
                },
                {
                    "label": " Disaster or Individual Tracking",
                    "value": "disaster_individual_tracking",
                },
                {"label": " Reset", "value": "reset"},
            ],
            value="reset",
            labelStyle={"display": "inline-block", "marginRight": "20px"},
        ),
    ]),
    html.Hr(),
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={
            "height": "auto",
            "minWidth": "100px",
            "width": "100px",
            "maxWidth": "180px",
            "whiteSpace": "normal",
        },
        row_selectable="single",
        selected_rows=[0],
    ),
    html.Br(),
    html.Hr(),
    html.Div(
        style={"display": "flex", "flexWrap": "wrap", "gap": "20px"},
        children=[
            html.Div(
                id="graph-id",
                style={"flex": "1", "minWidth": "450px", "maxHeight": "550px"},
            ),
            html.Div(
                id="map-id",
                style={"flex": "1", "minWidth": "450px"},
            ),
        ],
    ),
])

#############################################
# Interaction Between Components / Controller
#############################################

# MongoDB queries for rescue filter options (Dashboard Specifications Document)
WATER_RESCUE_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland",
        ]
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}
MOUNTAIN_WILDERNESS_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler",
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}
DISASTER_TRACKING_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler",
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
}


@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    """Update the data table based on the selected rescue filter."""
    if filter_type == "water_rescue":
        query = WATER_RESCUE_QUERY
    elif filter_type == "mountain_wilderness_rescue":
        query = MOUNTAIN_WILDERNESS_QUERY
    elif filter_type == "disaster_individual_tracking":
        query = DISASTER_TRACKING_QUERY
    else:
        query = {}

    records = db.read(query)
    data = pd.DataFrame.from_records(records)
    if "_id" in data.columns:
        data.drop(columns=["_id"], inplace=True)
    return data.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")],
)
def update_graphs(viewData):
    """Display breed distribution for the currently filtered data."""
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available to display.")

    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff["breed"].value_counts().head(10).reset_index()
    breed_counts.columns = ["breed", "count"]
    fig = px.pie(
        breed_counts,
        values="count",
        names="breed",
        title="Preferred Animals by Breed",
    )
    fig.update_layout(height=450, margin=dict(t=50, b=20, l=20, r=20))
    return [
        html.H3("Breed Distribution Chart"),
        dcc.Graph(figure=fig, style={"height": "450px"}),
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")],
)
def update_styles(selected_columns):
    """Highlight selected table columns."""
    if selected_columns is None:
        return []
    return [
        {"if": {"column_id": i}, "background_color": "#D2F3FF"}
        for i in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_viewport_data"),
    ],
)
def update_map(filtered_data, page_data):
    """Update the geolocation map for the filtered animals."""
    if filtered_data is None or len(filtered_data) == 0:
        return html.Div("No location data available.")

    if len(filtered_data) <= 100:
        map_data = filtered_data
        map_note = f"Showing {len(map_data)} animal location(s) for the selected filter."
    else:
        map_data = page_data or []
        map_note = f"Showing {len(map_data)} animal location(s) from the current table page."
    dff = pd.DataFrame.from_dict(map_data)
    markers = []
    for _, animal in dff.iterrows():
        markers.append(
            dl.Marker(
                position=[animal["location_lat"], animal["location_long"]],
                children=[
                    dl.Tooltip(animal["breed"]),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(animal["name"] if animal["name"] else "Unknown"),
                        html.P(f"Breed: {animal['breed']}"),
                    ]),
                ],
            )
        )

    return html.Div([
        html.H3("Geolocation Map"),
        html.P(map_note, style={"fontSize": "14px"}),
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[dl.TileLayer()] + markers,
        ),
    ])


# Run app and display result in JupyterLab mode
app.run(debug=True, jupyter_mode="external") 